### Preamble

In [1]:
from app.network.api import load_sumo_network
from app.network.road import build_lane_records, build_junction_records, compute_bounds, compute_edge_markings, compute_lane_markings
from app.network.parallel_marking import compute_parallel_direction_markings
from app.network.util import SVG

from pathlib import Path

%load_ext autoreload
%autoreload 2

## Road drawing steps

In [2]:
output_dir = "network-drawing"
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

### Cropping the network to a smaller area

In [3]:
root, road_file = load_sumo_network('tue-small')

# Construct lane records
lane_records = build_lane_records(root)
junction_records = build_junction_records(root)

# Collect the road polygons
lane_polys = [rec["polygon"] for rec in lane_records]
lane_lines = [rec["centerline"] for rec in lane_records]
junc_polys = [rec["polygon"] for rec in junction_records]

# Compute bounds for viewport fitting
bounds = compute_bounds(lane_polys + junc_polys)

# Compute lane markings
lane_markings = compute_lane_markings(lane_records)
edge_markings = compute_edge_markings(lane_polys + junc_polys)
parallel_markings, _ = compute_parallel_direction_markings(lane_records)

For a small part of the network, we are going to illustrate the full network drawing procedure, including the separating centerline algorithm. Therefore, we first filter the lane and junction records to a small area.

In [4]:
from shapely.geometry import box, Polygon

def crop_to_bounds(records, bounds):
    filtered_records = []
    for rec in records:
        if rec["polygon"].intersects(bounds):
            cropped_rec = dict(rec)  # Create a copy of the record
            cropped_rec["polygon"] = rec["polygon"].intersection(bounds)
            if "centerline" in rec:
                cropped_rec["centerline"] = rec["centerline"].intersection(bounds)
            if not isinstance(cropped_rec["polygon"], Polygon):
                continue  # Skip if the intersection is empty
            filtered_records.append(cropped_rec)
    
    return filtered_records

subset_bounds = { "minx": 280, "miny": -670, "maxx": 350, "maxy": -600 }
bounds_box = box(subset_bounds["minx"], subset_bounds["miny"], subset_bounds["maxx"], subset_bounds["maxy"])

# Draw cropping box on the full network
svg = SVG(bounds)
svg.draw_polygons(lane_polys + junc_polys)
svg.draw_polygons([bounds_box], stroke="green", stroke_width=0.5)
svg.write(output_dir / "tue-small.svg")

lane_records_cropped = crop_to_bounds(lane_records, bounds_box)
junction_records_cropped = crop_to_bounds(junction_records, bounds_box)

lane_polys_cropped = [rec["polygon"] for rec in lane_records_cropped]
lane_lines_cropped = [rec["centerline"] for rec in lane_records_cropped]
junc_polys_cropped = [rec["polygon"] for rec in junction_records_cropped]

# Compute lane markings
lane_markings_cropped = compute_lane_markings(lane_records_cropped)
edge_markings_cropped  = compute_edge_markings(lane_polys_cropped + junc_polys_cropped)
parallel_markings_cropped, _ = compute_parallel_direction_markings(lane_records_cropped)

### Drawing the steps

In [5]:
svg = SVG(subset_bounds)
svg.draw_polygons(junc_polys_cropped, stroke="#aaa", fill="#aaa")
svg.write(output_dir / "step1.svg")

svg.draw_lines(lane_lines_cropped, fill="#aaa", stroke_width=0.1)
svg.write(output_dir / "step2.svg")

svg.draw_polygons(lane_polys_cropped, stroke="#aaa", fill="#aaa")
svg.write(output_dir / "step3.svg")

svg.draw_polygons(edge_markings_cropped, stroke="white", stroke_width=0.25)
svg.write(output_dir / "step4.svg")

svg.draw_polygons(lane_markings_cropped, stroke="white", fill="white", stroke_width=0.05)
svg.write(output_dir / "step5.svg")

svg.draw_polygons(parallel_markings_cropped, stroke="#ffd84d", fill="#ffd84d", stroke_width=0.05)
svg.write(output_dir / "step6.svg")

## Separating centerline algorithm

In [6]:
output_dir = "parallel-marking"
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

### Step 1: Finding candidates

In [33]:
def base():
    svg = SVG(subset_bounds)
    svg.draw_polygons(lane_polys_cropped, stroke="grey", fill="grey", stroke_width=0.2)
    svg.draw_polygons(junc_polys_cropped, stroke="grey", fill="grey", stroke_width=0.2)
    return svg

svg = base()

ego = 2
svg.draw_polygons([lane_records_cropped[ego]["polygon"]], stroke="blue", fill="blue", stroke_width=0)
svg.write(output_dir / "tue-cropped-markings-1.svg")

In [8]:
# Compute lane markings
lane_markings = compute_lane_markings(lane_records_cropped)
edge_markings = compute_edge_markings(lane_polys_cropped + junc_polys_cropped)
parallel_markings, parallel_marking_debug = compute_parallel_direction_markings(lane_records_cropped)

In [ ]:
candidates = parallel_marking_debug["band_a"][ego]["candidates"]

# highlight all candidates
svg = base()
svg.draw_polygons([lane_records_cropped[j]["polygon"] for j in candidates], stroke="red", fill="red", stroke_width=0)
svg.write(output_dir / "tue-cropped-markings-2.svg")

# highlight one candidate
candidate = candidates[0]
svg.draw_polygons([lane_records_cropped[candidate]["polygon"]], stroke="purple", fill="purple", stroke_width=0)
svg.write(output_dir / "tue-cropped-markings-3.svg")

### Step 2: Buffer and compute intersection

In [40]:
svg = base()
band_1 = parallel_marking_debug["band_a"][ego]["polygon"]
band_2 = parallel_marking_debug["band_a"][candidate]["polygon"]
overlap = parallel_marking_debug["overlap"][2]["polygon"]
svg.draw_polygons([band_1], fill="red", stroke_width=0)
svg.draw_polygons([band_2], fill="red", stroke_width=0)
svg.draw_polygons([overlap], fill="blue", stroke_width=0)
svg.write(output_dir / "tue-cropped-markings-4.svg")